## הפרמטר הכי טוב: סריקה של χ²

בסעיף 12.1 הפרמטרים של המודל היו נתונים, ו-χ² מדד עד כמה המודל מתאים למדידות. כאן השאלה הפוכה: עבור איזה ערך של הפרמטר χ² הוא הקטן ביותר? ערך זה הוא **הפרמטר הכי טוב**: איתו המודל קרוב למדידות ככל האפשר, כאשר כל שארית נמדדת ביחס לשגיאת המדידה של הנקודה שלה.

המודל בעמוד זה הוא קו ישר שעובר דרך הראשית, עם פרמטר אחד בלבד:

$$R = a_1 \sin(2\theta)$$

לפי התאוריה $a_1 = v_0^2/g$, ועבור $v_0 = 20$ מ'/ש' ערכו כ-40.8 מטר. הדרך הפשוטה ביותר למצוא את $a_1$ מהנתונים היא **סריקה**: מחשבים את χ² עבור מועמדים רבים לערך של $a_1$, ושומרים את המועמד שנותן את ה-χ² הקטן ביותר.

הנתונים, והקוד שמכין אותם, הם אלה של סעיף 12.1. לכל זווית, ‎`x`‎ הוא $\sin(2\theta)$, ‎`y`‎ הוא הטווח הממוצע, ו-‎`dy`‎ היא שגיאת הממוצע: סטיית התקן חלקי השורש של מספר המדידות.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()
angles = sorted(df_clean["angle_deg"].unique())
x = np.array([np.sin(2*np.radians(a)) for a in angles])
y = np.array([df_clean[df_clean["angle_deg"] == a]["range_measured"].mean()
              for a in angles])
dy = np.array([
    df_clean[df_clean["angle_deg"] == a]["range_measured"].std(ddof=1)
    / np.sqrt(len(df_clean[df_clean["angle_deg"] == a]))
    for a in angles
])
a1_theory = 20.0**2 / g     # v0**2 / g (m)
print("y: ", np.round(y, 2))
print("dy:", np.round(dy, 2))

### χ² כסכום בלולאה

בנקודה ‎`i`‎ המודל חוזה את הערך ‎`a1 * x[i]`‎. χ² הוא סכום על כל הנקודות של ריבוע השארית חלקי השגיאה, ולכן אפשר לחשב אותו בלולאה עם משתנה צובר, כמו בלולאות של שבוע 4: מתחילים מ-0, ובכל סיבוב מוסיפים את התרומה של נקודה אחת.

In [ ]:
def chi2(a1, x, y, dy):
    """chi2 of the model y = a1 * x."""
    total = 0.0
    for i in range(len(x)):
        total += ((y[i] - a1 * x[i]) / dy[i])**2
    return total

print(f"{chi2(a1_theory, x, y, dy):.2f}")   # the theoretical a1
print(f"{chi2(45.0, x, y, dy):.2f}")

עבור הערך התאורטי χ² הוא כ-5.2, ועבור 45 הוא כבר כ-48.5. כדי למצוא את הערך שבו χ² מינימלי, מנסים מועמדים רבים.

### סריקה בלולאה

‎`np.linspace(35, 48, 131)`‎ נותן 131 מועמדים בצעד של 0.1. הלולאה עוברת עליהם ושומרת את המועמד הטוב ביותר עד כה. זה בדיוק המבנה של ‎`my_max`‎ משבוע 4: שם נשמר האיבר הגדול ביותר עד כה, וה-‎`if`‎ מחליף אותו כאשר מגיע איבר גדול יותר. כאן נשמר המועמד עם ה-χ² הקטן ביותר עד כה, וה-‎`if`‎ מחליף אותו כאשר מגיע מועמד עם χ² קטן יותר.

In [ ]:
a1_values = np.linspace(35, 48, 131)   # candidates, step 0.1

best_a1 = a1_values[0]
best_chi2 = chi2(best_a1, x, y, dy)
for a1 in a1_values:
    c = chi2(a1, x, y, dy)
    if c < best_chi2:
        best_a1 = a1
        best_chi2 = c

dof = len(x) - 1                       # 5 points, 1 parameter
print(f"best a1 = {best_a1:.1f}, chi2 = {best_chi2:.2f}")
print(f"chi2_red = {best_chi2 / dof:.2f}, dof = {dof}")

הפרמטר הכי טוב הוא $a_1 \approx 41.5$ מטר, קרוב לערך התאורטי 40.8. למודל פרמטר אחד בלבד, ולכן מספר דרגות החופש הוא ‎`5 - 1 = 4`‎. χ² המצומצם, כ-0.92, קרוב ל-1: השאריות בגודל שמתאים לשגיאות המדידה. הדיוק של הסריקה מוגבל בצעד שבין המועמדים, כאן 0.1.

### אותה סריקה במערך

את שתי הלולאות, על המועמדים ועל הנקודות, אפשר להחליף בפעולות על מערכים. ‎`a1_values[:, None]`‎ הוא עמודה בצורה ‎`(131, 1)`‎, ו-‎`x[None, :]`‎ הוא שורה בצורה ‎`(1, 5)`‎. לפי כלל 3 משבוע 9 (broadcasting), המכפלה שלהם היא מערך ‎`model`‎ בצורה ‎`(131, 5)`‎: שורה לכל מועמד ועמודה לכל נקודה, ובכל תא ערך המודל של מועמד אחד בנקודה אחת. מכאן:

1. ‎`(y - model) / dy`‎ הן השאריות חלקי השגיאות, באותה צורה. ‎`y`‎ ו-‎`dy`‎ הם בצורה ‎`(5,)`‎, ולכן הם מתאימים לציר האחרון.
2. ‎`.sum(axis=1)`‎ מסכם על ציר הנקודות. לפי כלל 2 משבוע 9 הציר הזה נעלם, ונשאר χ² אחד לכל מועמד.
3. ‎`.argmin()`‎ מחזיר את המיקום של ה-χ² הקטן ביותר, ובמיקום הזה נמצא המועמד הטוב ביותר.

In [ ]:
model = a1_values[:, None] * x[None, :]    # (candidates, points)
chi2_values = (((y - model) / dy)**2).sum(axis=1)
i_best = chi2_values.argmin()
print(model.shape, chi2_values.shape)
print(f"best a1 = {a1_values[i_best]:.1f}, chi2 = {chi2_values[i_best]:.2f}")

התוצאה זהה לזו של הלולאה.

### הגרף של χ² כפונקציה של הפרמטר

הגרף מציג את ‎`chi2_values`‎ מול ‎`a1_values`‎. המינימום מסומן בעיגול, והערך התאורטי בריבוע.

In [ ]:
fig, ax = plt.subplots()
ax.plot(a1_values, chi2_values)
ax.plot(a1_values[i_best], chi2_values[i_best], "o", label="Minimum")
ax.plot(a1_theory, chi2(a1_theory, x, y, dy), "s", label="Theory")
ax.set_xlabel("a1 (m)")
ax.set_ylabel("χ²")
ax.legend()
plt.show()

χ² יורד עד המינימום ועולה אחריו, ולכן יש לו מינימום אחד ברור. המודל ליניארי בפרמטר $a_1$, ולכן העקומה היא פרבולה. בסעיף 12.3 נוסחה סגורה, נוסחת הריבועים הפחותים, נותנת את שני הפרמטרים של קו ישר ישירות, בלי סריקה.

### באג נפוץ: סכום על הציר הלא נכון

במערך ‎`model`‎ ציר 0 הוא ציר המועמדים וציר 1 הוא ציר הנקודות. ‎`.sum(axis=0)`‎ מסכם על המועמדים ונותן 5 מספרים, אחד לכל נקודה. ‎`argmin()`‎ של המערך הזה מחזיר מיקום של נקודה, מספר בין 0 ל-4, והקוד משתמש בו כמיקום של מועמד. שגיאה אין: מתקבל מספר בתחום הסריקה, אבל הוא שגוי. בדיקה פשוטה היא להדפיס את הצורה: במערך של χ² צריך להיות ערך אחד לכל מועמד.

In [ ]:
wrong = (((y - model) / dy)**2).sum(axis=0)   # sums over the candidates
print(wrong.shape)                            # one value per point
print(f"{a1_values[wrong.argmin()]:.1f}")     # not the best a1

### נסו בעצמכם

שלוש נקודות: ‎`x = [1, 2, 3]`‎ ו-‎`y = [2, 4, 7]`‎, עם שגיאה 1 בכל נקודה. שלושה מועמדים לשיפוע: ‎`1.5`‎, ‎`2.0`‎ ו-‎`2.5`‎. חשבו קודם בראש את χ² של כל מועמד: כאשר השגיאה היא 1, χ² הוא פשוט סכום ריבועי השאריות. אחר כך השלימו את ‎`chi2_scan`‎ בלי לולאה, כמו בסריקה במערך שלמעלה, הריצו, ובדקו שהתוצאה מתאימה לחישוב שלכם. לבסוף מצאו את המועמד הטוב ביותר בעזרת ‎`argmin()`‎.

In [ ]:
def chi2_scan(a1_values, x, y, dy):
    """chi2 of the model y = a1 * x, one value per candidate a1."""
    pass   # replace with your code

x3 = np.array([1.0, 2.0, 3.0])
y3 = np.array([2.0, 4.0, 7.0])
dy3 = np.array([1.0, 1.0, 1.0])
candidates = np.array([1.5, 2.0, 2.5])
print(chi2_scan(candidates, x3, y3, dy3))

`````{admonition} פתרון
:class: dropdown, tip
```python
def chi2_scan(a1_values, x, y, dy):
    """chi2 of the model y = a1 * x, one value per candidate a1."""
    model = a1_values[:, None] * x[None, :]
    return (((y - model) / dy)**2).sum(axis=1)

chi2_3 = chi2_scan(candidates, x3, y3, dy3)
print(chi2_3)                         # [7.5 1.  1.5]
print(candidates[chi2_3.argmin()])    # 2.0
```
בחישוב בראש: עבור 1.5 השאריות הן ‎`0.5, 1, 2.5`‎, וסכום הריבועים הוא ‎`0.25 + 1 + 6.25 = 7.5`‎. עבור 2.0 השאריות הן ‎`0, 0, 1`‎, והסכום הוא 1. עבור 2.5 השאריות הן ‎`-0.5, -1, -0.5`‎, והסכום הוא 1.5. המועמד הטוב ביותר הוא 2.0.
`````

### בדקו את עצמכם

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "בסריקה בלולאה, מתי מחליפים את המועמד הטוב ביותר עד כה?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "כאשר ה-χ² של המועמד הנוכחי קטן מה-χ² הטוב ביותר עד כה",
                "correct": true,
                "feedback": "נכון: מחפשים מינימום, ולכן שומרים את הקטן ביותר, כמו ש-my_max שומר את הגדול ביותר."
            },
            {
                "answer": "כאשר ה-χ² של המועמד הנוכחי גדול מה-χ² הטוב ביותר עד כה",
                "correct": false,
                "feedback": "לא: כך נשמר בסוף המועמד הגרוע ביותר. בסריקה ההשוואה הפוכה לזו של my_max."
            },
            {
                "answer": "כאשר ה-χ² של המועמד הנוכחי קרוב ל-1",
                "correct": false,
                "feedback": "לא: הסריקה מחפשת את ה-χ² הקטן ביותר. רק χ² המצומצם של המועמד הטוב ביותר אמור להיות קרוב ל-1."
            },
            {
                "answer": "בכל סיבוב של הלולאה",
                "correct": false,
                "feedback": "לא: אז בסוף הלולאה נשאר המועמד האחרון, ולא הטוב ביותר."
            }
        ]
    },
    {
        "question": "במערך המודל של הסריקה יש שורה לכל מועמד ועמודה לכל נקודה. על איזה ציר סוכמים את ריבועי השאריות המנורמלות כדי לקבל χ² לכל מועמד?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "על ציר 1, ציר הנקודות",
                "correct": true,
                "feedback": "נכון: ציר הנקודות נעלם, ונשאר ערך אחד לכל מועמד."
            },
            {
                "answer": "על ציר 0, ציר המועמדים",
                "correct": false,
                "feedback": "לא: כך מתקבל ערך אחד לכל נקודה. זה הבאג הנפוץ של העמוד."
            },
            {
                "answer": "על שני הצירים יחד",
                "correct": false,
                "feedback": "לא: כך מתקבל מספר אחד בלבד, הסכום על כל המועמדים וכל הנקודות."
            },
            {
                "answer": "אין צורך לסכום, כי argmin מוצא את המינימום במערך כולו",
                "correct": false,
                "feedback": "לא: בלי סכום, argmin מוצא את השארית הקטנה ביותר של נקודה אחת, ולא את ה-χ² הקטן ביותר של מועמד."
            }
        ]
    },
    {
        "question": "בסריקה יש 5 נקודות, ולמודל פרמטר אחד. מהו מספר דרגות החופש?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "4",
                "correct": true,
                "feedback": "נכון: מספר הנקודות פחות מספר הפרמטרים, 5 פחות 1."
            },
            {
                "answer": "3",
                "correct": false,
                "feedback": "לא: 3 מתאים למודל עם שני פרמטרים, כמו בסעיף 12.1."
            },
            {
                "answer": "5",
                "correct": false,
                "feedback": "לא: 5 הוא מספר הנקודות. מחסרים ממנו את מספר הפרמטרים."
            },
            {
                "answer": "131",
                "correct": false,
                "feedback": "לא: 131 הוא מספר המועמדים בסריקה, והוא אינו קשור לדרגות החופש."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)